In [ ]:
# import necessary packages
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import matplotlib.pyplot as plt

This notebook is dependent on having run disubstituted_substituent_selection_preparation_results.ipynb, at least through the point at which 'selected_3_4_substituted_nitriles.csv' is created. No quantum mechanical calculations are necessary to reach this point.

In [ ]:
# read in the 3,4-disubstituted nitriles
selected_3_4 = 'selected_3_4_substituted_nitriles.csv'

df_3_4 = pd.read_csv(selected_3_4)

In [ ]:
# make mol objects for the substituents
smis = df_3_4['SMILES']
mol_objects = [Chem.MolFromSmiles(smi) for smi in smis]
df_3_4['mols'] = mol_objects

In [ ]:
# identify the number of the meta substituent
df_3_4['meta_number'] = df_3_4['Name'].str.split('_').str[0].astype(int)
df_3_4['meta_number'] = df_3_4['meta_number'].apply(lambda x: f"{int(x):03d}")

In [ ]:
# define a SMARTS reaction to duplicate the meta substituent to the other meta position
rxn_3_4_5 = AllChem.ReactionFromSmarts('[N:1]#[C:2]-[c:3]1[c:4][c:5][c:6]([c:7]([c:8]1)-[*:9])-[*:10]>>[N:1]#[C:2][c3]1[c8][c7]([c6]([c5]([c4]1)[*:9])[*:10])[*:9]')
rxn_3_4_5

In [ ]:
# generate the trisubstituted SMILES
products_3_4_5 = []
for subst_mol in df_3_4['mols']:
    print(f"Reacting benzonitrile with {subst_mol}")
    result = rxn_3_4_5.RunReactants((subst_mol,))
    print(f"Result: {result}")
    if result and result[0]:  # Check if there is at least one product
        products_3_4_5.append(result[0][0])
    else:
        products_3_4_5.append(None)  

smis_3_4_5 = [Chem.MolToSmiles(mol) if mol is not None else '' for mol in products_3_4_5]
df_3_4['smiles_3_4_5'] = smis_3_4_5

In [ ]:
df_3_4_5 = df_3_4.copy()
df_3_4_5.drop(columns=['mols','SMILES'], inplace=True)

In [ ]:
# add the additional meta parameter to the hammett sum
df_3_4_5['sum_3_4_5'] = df_3_4_5['hammett_sum'] + df_3_4_5['sigma_m']
df_3_4_5.drop(columns=['hammett_sum'], inplace=True)
df_3_4_5.rename(columns={'sum_3_4_5':'hammett_sum'}, inplace=True)

In [ ]:
# create descriptive file names
df_3_4_5['code_name'] = df_3_4_5.apply(
    lambda row: f"meta{row['Name'].split('_')[0]}_para{row['Name'].split('_')[1]}_meta{row['meta_number']}_nitrile", axis=1
)

In [ ]:
df_3_4_5.drop(columns=['Name', 'meta_number'], inplace=True)
df_3_4_5.to_csv('selected_3_4_5_substituted_nitriles.csv', index=False)

In [ ]:
# create csvs for aqme inputs
trisubstituted = 'selected_3_4_5_substituted_nitriles.csv'

aqme_3_4_5 = pd.read_csv(trisubstituted)
aqme_3_4_5.drop(columns=['sigma_m','sigma_p','hammett_sum'], inplace=True)

aqme_3_4_5.to_csv('aqme_3_4_5.csv', index=False)

In [ ]:
# generate conformers and optimization inputs
! python -m aqme --csearch --program rdkit --input aqme_3_4_5.csv 
! mv CSEARCH conformers_3_4_5
! python -m aqme --qprep --program gaussian --qm_input "B3LYP def2SVP emp=GD3BJ opt freq" --files "conformers_3_4_5/*.sdf"
! mv QCALC optimization_inputs_3_4_5

In [ ]:
# check Gaussian output files for errors using aqme
! python -m aqme --qcorr --program gaussian --files "optimization_outputs_3_4_5/*.log"

In [ ]:
# imaginary frequencies > -7 cm-1 were accepted; this affected meta372_para125_meta372_nitrile_5_rdkit_conf_2, meta167_para307_meta167_nitrile_5_rdkit_conf_3, and meta297_para129_meta297_nitrile_5_rdkit_conf_5

In [ ]:
# prepare single point files to obtain charges
! python -m aqme --qprep --program gaussian --qm_input "B3LYP def2QZVPP emp=GD3BJ pop=(Hirshfeld)" --files "optimization_outputs_3_4_5/success/*.log"
! mv QCALC single_point_inputs_3_4_5

In [ ]:
# check Gaussian output files for errors using aqme
! python -m aqme --qcorr --program gaussian --files "single_point_outputs_3_4_5/*.log"

In [ ]:
# OPTIONAL – make new directories and move files
! mkdir spc_3_4_5
! cp single_point_outputs_3_4_5/success/SP_calcs/*.log spc_3_4_5/
! mkdir opt_3_4_5
! cp optimization_outputs_3_4_5/success/*.log opt_3_4_5/

In [ ]:
# CHANGE ENVIRONMENT TO APPROPRIATE ENVIRONMENT FOR MOLDSCRIPT  
# use molDscript to parse atomic charges 
! python -m moldscript --opt opt_3_4_5 --spc spc_3_4_5 --charges spc_3_4_5 --substructure 'cC#N' --boltz --min_max --lowe
! mkdir charges_3_4_5
! mv *_level.csv charges_3_4_5
! mv MOLDSCRIPT*.dat charges_3_4_5

In [ ]:
# clean molDscript data to obtain only the desired charges
trisub = 'charges_3_4_5/lowest_energy_atom_level.csv'
df_3_4_5 = pd.read_csv(trisub)

df_3_4_5_N = df_3_4_5[df_3_4_5['atom_type'] == 'N']

df_3_4_5_N = df_3_4_5_N.sort_values(by='filename')
df_3_4_5_N = df_3_4_5_N.rename(columns={'filename': 'Name'})
df_3_4_5_N['Name'] = df_3_4_5_N['Name'].apply(lambda x: '_'.join([x.split('_')[0], x.split('_')[1], x.split('_')[2], x.split('_')[3].zfill(4)]))

In [ ]:
# identify benzonitriles for which multiple N charges were parsed
repeats_3_4_5 = df_3_4_5_N[df_3_4_5_N.duplicated('Name', keep=False)]
repeats_3_4_5_df = repeats_3_4_5.copy()
repeats_3_4_5_df

In [ ]:
# remove incorrectly parsed N charges from the dataset after manual inspection
df_3_4_5_N = df_3_4_5_N[~((df_3_4_5_N['Name'] == 'meta003_para125_meta003_nitrile') & (df_3_4_5_N['atom_index'] != 1))]
df_3_4_5_N = df_3_4_5_N[~((df_3_4_5_N['Name'] == 'meta011_para031_meta011_nitrile') & (df_3_4_5_N['atom_index'] != 1))]
df_3_4_5_N = df_3_4_5_N[~((df_3_4_5_N['Name'] == 'meta075_para008_meta075_nitrile') & (df_3_4_5_N['atom_index'] != 1))]

In [ ]:
# add the experimental Hammett sums
trisub_values = 'selected_3_4_5_substituted_nitriles.csv'
exp_3_4_5 = pd.read_csv(trisub_values)

exp_3_4_5.rename(columns={'code_name': 'Name'}, inplace=True)
exp_3_4_5 = exp_3_4_5[['Name', 'hammett_sum','sigma_m','sigma_p']]
compare_3_4_5 = pd.merge(df_3_4_5_N, exp_3_4_5, on='Name', how='inner')

In [ ]:
# predict hammett parameters based on the monosubstituted model
compare_3_4_5['predicted_hammett'] = 34.53 * compare_3_4_5['cm5_charge'] + 11.90

In [ ]:
compare_3_4_5.to_csv('hammett_predictions_3_4_5.csv')